# Transformer

In [ ]:
import torch

## Basic Modules

- Positional Encoding
- FNN
- MHA


**Positional Encoding**

$$
PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{\text{model}}}) \\
PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{\text{model}}})
$$

Note the division term $\frac{1}{10000^{\frac{2i}{d}}} = 10000^{-\frac{2i}{d}}$ is usually implimented to avoid using `pow()` for efficiency improvement. Because $a^b = e^{b\ln a}$, thus

$$10000^{-\frac{2i}{d}} = e^ {-\frac{2i}{d}\ln 10000}$$

The benefits are:
- Numerical Stability: Operations in log-space are generally more numerically stable. Converting the power operation $a^b$ into $e^{b\ln a}$ avoids potential underflow or precision issues that can occur with direct exponentiation `pow()`, especially when gradients are backpropagated. 
    - When training neural networks, gradients are backpropagated. The derivative of $x^n$ involves $x^{n-1}$, which can be volatile. 
    - The exponential function $e^x$ is its own derivative. Working in log-space smooths out the gradients, preventing them from exploding or vanishing as easily as they might with direct high-degree power functions.
- Performance: It transforms a geometric progression (the frequencies) into an arithmetic progression in the exponent. Modern hardware (GPUs) is extremely efficient at computing linear vector arithmetic (multiplication) followed by an element-wise exp, often more so than calculating a generic element-wise `pow` function.

In [1]:

def sinusoidal_pe(d_model, max_len = 5000, dominator = 10000.0):
    """
    Generate sinusoidal positional encoding.

    Returns:
        pe: Positional encoding tensor of shape (max_len, d_model) 
    """
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(dominator)) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    pe = pe.unsqueeze(0).transpose(0, 1)
    
    return pe

**FNN**

Feedforward neural network

In [ ]:
class FNN(torch.nn.Module):
    def __init__(self, in_features, hidden_features, out_features = None, bias = True):
        super(FNN, self).__init__()
        out_features = out_features or in_features
        self.linear1 = torch.nn.Linear(in_features, hidden_features, bias)
        self.activation = torch.nn.ReLU()
        self.linear2 = torch.nn.Linear(hidden_features, out_features, bias)
    
    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        return x

**Multi-head Attention**
Multihead attention



In [ ]:
class MHA(torch.nn.Module):
    def __init__(self, emb_dim, num_heads, dropout = 0.0):
        super(MHA, self).__init__()
        # make sure emb_dim is divisible by num_heads
        assert emb_dim % num_heads == 0, "Embedding dimension must be divisible by number of heads"

        self.emb_dim = emb_dim
        self.num_heads = num_heads
        self.head_dim = emb_dim // num_heads

        self.query_linear = torch.nn.Linear(emb_dim, emb_dim)
        self.key_linear = torch.nn.Linear(emb_dim, emb_dim)
        self.value_linear = torch.nn.Linear(emb_dim, emb_dim) 
        self.out_linear = torch.nn.Linear(emb_dim, emb_dim)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, query, key = None, value = None, mask = None):
        """
        Args:
            query: Tensor of shape (batch_size, seq_len, emb_dim)
            key: Tensor of shape (batch_size, seq_len, emb_dim) (optional)
            value: Tensor of shape (batch_size, seq_len, emb_dim) (optional)
            mask: Tensor of shape (batch_size, seq_len, seq_len) (optional) 
        """
        if key is None:
            key = query
        if value is None:
            value = query

        # project query, key, value
        query = self.query_linear(query)  # (batch_size, seq_len, emb_dim)
        key = self.key_linear(key)        # (batch_size, seq_len, emb_dim)
        value = self.value_linear(value)  # (batch_size, seq_len, emb_dim) 

        # split into heads
        batch_size = query.size(0)
        query = query.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)  # (batch_size, num_heads, seq_len, head_dim)
        key = key.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)      # (batch_size, num_heads, seq_len, head_dim)
        value = value.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)  # (batch_size, num_heads, seq_len, head_dim)

        # scaled dot-product attention
        attn_scores = query @ key.transpose(-2, -1) / (self.head_dim ** 0.5)  # (batch_size, num_heads, seq_len, seq_len)  
        
        # apply mask if provided
        if mask is not None:
            # shape assertion
            assert mask.size() == (batch_size, query.size(2), key.size(2)), "Mask shape must be (batch_size, seq_len, seq_len)"
            #attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))
            attn_scores = mask[:, None, :, :] * attn_scores + (1 - mask[:, None, :, :]) * float('-inf')  # (batch_size, num_heads, seq_len, seq_len)

        # softmax to get attention weights
        attn_weights = torch.nn.Softmax(attn_scores, dim=-1)  # (batch_size, num_heads, seq_len, seq_len)

        # apply dropout to attention weights
        attn_weights = self.dropout(attn_weights)

        # weighted sum of values
        attn_output = attn_weights @ value  # (batch_size, num_heads, seq_len, head_dim)

        # concatenate heads
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.emb_dim)  # (batch_size, seq_len, emb_dim)
        attn_output = self.out_linear(attn_output)  # (batch_size, seq_len, emb_dim)

        return attn_output

## Basic Blocks

**Dropout**
Dropout are used in several places to prevent overfitting and stablizing training.

1. On input embeddings: dropout is applied to the sum of token embeddings + positional encodings before feeding into the first encoder/decoder layer
2. After attention weights: inside the multi-head attention mechanism, dropout is applied to the softmax of attention scores before multiplying by the values of V
3. After each sublayer's output
- Output of the multi-head attention block
    ```
    x = LayerNorm(x + Dropout(Attention(x)))
    ```
- Output of the feed-forward network
    ```
    x = LayerNorm(x + Dropout(FeedForward(x)))
    ```
**Attentions**
The default multi-head attention has a time complexity of $O(N^2d)$ and memory complexity of $O(N^2)$. 
The learned attention has a low-rank property, which can be exploited to reduce the complexity.

Many methods have been proposed to reduce the complexity of self-attention, including:

- Sparse Attention: Only attend to a subset of the input tokens, reducing the number of computations.
- Kernelized Attention: Use kernel methods to approximate the softmax attention mechanism.
- Linear Attention: Reformulate the attention mechanism to have linear complexity with respect to the sequence length.

**Encoder Block**


In [ ]:
class EncoderBlock(torch.nn.Module):
    def __init__(self, emb_dim, num_heads, ffn_hidden_dim, dropout = 0.0):
        super(EncoderBlock, self).__init__()
        self.mha = MHA(emb_dim, num_heads, dropout)
        self.ffn = FNN(emb_dim, ffn_hidden_dim)
        self.norm1 = torch.nn.LayerNorm(emb_dim)
        self.norm2 = torch.nn.LayerNorm(emb_dim)
        self.dropout1 = torch.nn.Dropout(dropout)
        self.dropout2 = torch.nn.Dropout(dropout)
    
    def forward(self, x, mask = None):
        # multi-head attention: self-attention
        attn_output = self.mha(x, x, x, mask)  # (batch_size, seq_len, emb_dim)

        # add & norm
        x = self.norm1(x + self.dropout1(attn_output))  # (batch_size, seq_len, emb_dim)
  
        # feed-forward network
        ffn_output = self.ffn(x)  # (batch_size, seq_len, emb_dim)
        # add & norm
        x = self.norm2(x + self.dropout2(ffn_output))  # (batch_size, seq_len, emb_dim)

        return x

## Encoder/Decoder